In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import rq4

# Notebook 10's run, then the reruns of the Runs that apir-adapter fixes changed: notebook 12 (S3 and M1, commit
# a053272) and notebook 13 (S3 again, commit after it). A Run in a later run replaces the Run of the same name in an
# earlier one, as a resumed Run's latest attempt does. analysis/rq4.py holds the loading, tables and figures, which
# `make figures RECOVERY=...` also draws.
RAW = repo / "results/raw"
RUNS = [RAW / "rq4-day-20261111", RAW / "rq4-rerun-20261111", RAW / "rq4-rerun-s3-20261111"]
df, manifest, source, metas = rq4.load(RUNS)
for meta in metas:
    print(f"run {meta['run_id']} at {meta['git_sha'][:7]}, citable={meta['citable']}, dirty={meta['dirty']}")
source = {name: d.name for name, d in source.items()}
not_ok = {n: m["outcome"] for n, m in manifest.items() if m["outcome"] != "ok"}
print(f"{len(manifest)} Runs, {len(not_ok)} not ok; {len(df)} records after warm-ups")
for d, n in pd.Series(source).value_counts().items():
    print(f"  {n:3d} Runs from {d}")
for n, o in not_ok.items():
    print(f"  {o}: {n}")


run rq4-day-20261111 at 29bcc2a, citable=True, dirty=False
run rq4-rerun-20261111 at a053272, citable=True, dirty=False
run rq4-rerun-s3-20261111 at b353acc, citable=True, dirty=False
116 Runs, 0 not ok; 46912 records after warm-ups
  100 Runs from rq4-day-20261111
    8 Runs from rq4-rerun-s3-20261111
    8 Runs from rq4-rerun-20261111


In [2]:
RNG = np.random.default_rng(rq4.SEED)  # one generator, drawn in the order cells 3 and 5 use it

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (max(0.0, c - h), min(1.0, c + h))

VERIFIABLE = rq4.VERIFIABLE

# Detection: the first answer of every faulted trial, whatever the policy, and every faulted offline repetition.
# A baseline has no check, so it detects nothing by construction and gets no rate. Its drop_server answers abort
# because nothing arrives, not because a check failed.
pd.set_option("display.width", 200)
first = rq4.first_answers(df)
det = (first.groupby(["arm", "fault"]).agg(n=("detected", "size"), detected=("detected", "sum"),
                                            silent_wrong=("outcome", lambda o: int((o == "silent_wrong").sum())),
                                            aborted=("outcome", lambda o: int((o == "aborted").sum())))
       .reset_index())
off = df[(df.kind == "offline") & df.faulted.astype(bool)]
det = pd.concat([det, off.groupby(["arm", "fault"]).agg(n=("detected", "size"), detected=("detected", "sum"))
                 .reset_index()])
verifiable = det.arm.isin(VERIFIABLE)
det.loc[~verifiable, "detected"] = np.nan
det["rate"] = det.detected / det.n
det[["lo95", "hi95"]] = [wilson(k, n) if v else (np.nan, np.nan) for k, n, v in zip(det.detected, det.n, verifiable)]
print("Detection of faulted first answers (trials) and faulted offline phases, Wilson 95 % CI "
      "(n/a: baselines have no check; offline phases have no outcome)")
print(det.round(3).to_string(index=False, na_rep="n/a"))

# False rejects: an honest first answer the client rejected, over every sweep query.
honest = df[(df.kind == "sweep") & ~df.faulted.astype(bool)]
fr = honest.groupby("arm").agg(honest=("detected", "size"), rejected=("detected", "sum"))
print("\nFalse rejects among honest sweep answers (spec: should never happen)")
print(fr.to_string())

Detection of faulted first answers (trials) and faulted offline phases, Wilson 95 % CI (n/a: baselines have no check; offline phases have no outcome)
   arm           fault   n detected  silent_wrong  aborted rate  lo95  hi95
    M0     drop_server 100      n/a           0.0    100.0  n/a   n/a   n/a
    M0 flip_answer_bit 100      n/a         100.0      0.0  n/a   n/a   n/a
    M0    stale_record 100      n/a         100.0      0.0  n/a   n/a   n/a
    M0    wrong_record 100      n/a         100.0      0.0  n/a   n/a   n/a
    M1     drop_server 200      200           0.0    100.0  1.0 0.981   1.0
    M1 flip_answer_bit 200      200           0.0    100.0  1.0 0.981   1.0
    M1    stale_record 200      200           0.0    100.0  1.0 0.981   1.0
    M1    wrong_record 200      200           0.0    100.0  1.0 0.981   1.0
S0-cpp     drop_server 100      n/a           0.0    100.0  n/a   n/a   n/a
S0-cpp flip_answer_bit 100      n/a         100.0      0.0  n/a   n/a   n/a
S0-cpp    stal

In [3]:
# F12: what a rejection costs, per fault and policy, as a multiple of the arm's fault-free latency
# (spec 6, recovery_cost_ratio). Trials give abort and switch_replica for every fault. re_preprocess comes from
# the f = 0.1 sweep (wrong_record), where a rejected query is followed by a fresh offline phase (rq4.f12_table).
base, f12 = rq4.f12_table(df, RNG)
print("Fault-free latency and bytes per arm (median of the f = 0 sweep control)")
print(base.to_string())
print("\nTime to a verified record or to abort, as a multiple of the fault-free latency (median, bootstrap 95 % CI)")
print(f12.round(3).to_string(index=False))


Fault-free latency and bytes per arm (median of the f = 0 sweep control)
        latency_s      bytes
arm                         
M0       0.025453   524418.0
M1       0.037156   807554.0
S0-cpp   0.031894   109184.0
S0-go    0.015382    86700.0
S1       0.032296   109184.0
S2       0.042275   131072.0
S3       1.987670  3408088.0
S4       1.282885    47850.0

Time to a verified record or to abort, as a multiple of the fault-free latency (median, bootstrap 95 % CI)
arm                 fault         policy   n    ratio  ratio_lo  ratio_hi  time_s  extra_bytes_ratio  extra_lo  extra_hi  recovered  aborted  silent_wrong
 S1       flip_answer_bit          abort 100    0.853     0.851     0.854   0.028              0.000     0.000     0.000          0      100             0
 S1       flip_answer_bit switch_replica 100    1.844     1.842     1.846   0.060              1.000     1.000     1.000        100        0             0
 S1          wrong_record          abort 100    0.851     0.849 

In [4]:
# F12, drawn by rq4.plot_f12 in the reference palette (dataviz skill); re_preprocess points carry direct labels.
FIG = repo / "docs/figures"
print([p.name for p in rq4.save(rq4.plot_f12(f12), FIG, "F12_recovery_cost")])


['F12_recovery_cost.png', 'F12_recovery_cost.pdf']


In [5]:
# F13: expected cost per query against the cheating rate f, measured in the sweep against computed from component
# costs measured outside it (spec 6.1; the model and its components are in rq4.f13_tables).
f13, curves, components = rq4.f13_tables(df, RNG)
for arm, parts in components.items():
    print(f"{arm:3s} components: " + ", ".join(f"{k} {v:.4g}" + ("" if k == "d" else " s") + f" (n={n})"
                                              for k, (v, n) in parts.items()))
print("\nMean time per query (s): measured in the sweep against computed from components, 95 % CIs; "
      "agree = the two intervals overlap")
print(f13.round(4).to_string(index=False))
checked = f13[f13.f > 0]          # f = 0 is the control that h comes from
print("\nAt f > 0, per policy: measured points, how many agree, and the largest |measured / computed - 1|")
print(checked.groupby("policy", sort=False).agg(points=("agree", "size"), agree=("agree", "sum"),
                                                 max_rel_err=("rel_err", lambda e: e.abs().max())).round(4).to_string())


S1  components: h 0.03233 s (n=1000), r 0.02759 s (n=100), s 0.03194 s (n=100), o 137 s (n=3), d 1 (n=200)
S2  components: h 0.04232 s (n=1000), r 0.03707 s (n=100), s 0.04237 s (n=100), o 196.8 s (n=3), d 1 (n=200)
S3  components: h 2.011 s (n=50), r 1.96 s (n=30), s 2.002 s (n=30), o 0 s (n=0), d 1 (n=60)
S4  components: h 1.284 s (n=50), r 1.246 s (n=30), s 1.286 s (n=30), o 0 s (n=0), d 1 (n=60)
M1  components: h 0.03721 s (n=1000), r 0.0374 s (n=100), s 0.03707 s (n=100), o 0 s (n=0), d 1 (n=200)

Mean time per query (s): measured in the sweep against computed from components, 95 % CIs; agree = the two intervals overlap
arm         policy    f    n  measured    m_lo    m_hi  computed    c_lo    c_hi  rel_err  agree
 S1          abort 0.00 1000    0.0323  0.0323  0.0323    0.0323  0.0323  0.0323   0.0000   True
 S1          abort 0.01 1000    0.0323  0.0322  0.0323    0.0323  0.0323  0.0323  -0.0003   True
 S1          abort 0.10 1000    0.0318  0.0317  0.0319    0.0319  0.0318  0.

In [6]:
# F13, drawn by rq4.plot_f13: measured points over the computed curve, per arm and policy.
print([p.name for p in rq4.save(rq4.plot_f13(f13, curves), FIG, "F13_expected_cost")])


['F13_expected_cost.png', 'F13_expected_cost.pdf']


In [7]:
# What the reruns changed: each rerun Run in every run that measured it, notebook 10 first.
def run_summary(run_dir, name):
    m = next(json.loads(l) for l in (run_dir / "manifest.jsonl").read_text().splitlines()
             if json.loads(l)["name"] == name)
    rs = [json.loads(l) for l in (run_dir / f"{name}.jsonl").read_text().splitlines()]
    rs = [r for r in rs if not r["is_warmup"]]
    honest = [r for r in rs if not r["faulted"]]
    per_attempt = [r["metrics"]["recovery_s"] / r["recovery_attempts"] for r in rs if r["recovery_attempts"]]
    return dict(query_gen_s=np.median([r["metrics"]["query_gen_s"] for r in rs]),
                honest_query_gen_s=np.median([r["metrics"]["query_gen_s"] for r in honest]) if honest else np.nan,
                recovery_per_attempt_s=np.median(per_attempt) if per_attempt else np.nan,
                peak_gib=m["peak_footprint_bytes"] / 2**30)

NOTEBOOK = {RUNS[0].name: "nb 10", RUNS[1].name: "nb 12", RUNS[2].name: "nb 13"}
rows = []
for name in sorted(n for n, d in source.items() if d != RUNS[0].name):
    for run_dir in RUNS:
        if (run_dir / f"{name}.jsonl").exists():
            rows.append(dict(run=name.replace("-db134217728", "").replace("-wrong_record", ""),
                             source=NOTEBOOK[run_dir.name], **run_summary(run_dir, name)))
cmp = pd.DataFrame(rows).pivot(index="run", columns="source")
print("Medians per Run in each run that measured it (the last one is analysed); peak footprint per Run")
print(cmp.round(4).to_string(na_rep="-"))

Medians per Run in each run that measured it (the last one is analysed); peak footprint per Run
                                                   query_gen_s                 honest_query_gen_s                 recovery_per_attempt_s                 peak_gib                
source                                                   nb 10   nb 12   nb 13              nb 10   nb 12   nb 13                  nb 10   nb 12   nb 13    nb 10   nb 12   nb 13
run                                                                                                                                                                              
sweep-M1-rb1024-switch_replica-p0_01-cheat              0.0000  0.0000       -             0.0000  0.0000       -                 0.0736  0.0372       -   0.9879  1.0011       -
sweep-M1-rb1024-switch_replica-p0_1-cheat               0.0000  0.0000       -             0.0000  0.0000       -                 0.0742  0.0372       -   1.0944  1.0235       -
sweep-M1-rb102